# ЛР-2. Кластеризация

Датасет: Chemical Composition of Ceramic Samples (UCI 583).  
Алгоритмы: K-means++ и DBSCAN (реализация с нуля).  
Метрики: Silhouette Score, Adjusted Rand Index.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from mlcore.data.datasets import load_lr2_dataset
from mlcore.preprocessing.scaling import standardize
from mlcore.preprocessing.encoding import label_encode
from mlcore.tabular.preprocessing import normalize_missing_values
from mlcore.utils.timing import timer

ROOT = Path.cwd()
for p in [ROOT, ROOT.parent, ROOT.parent.parent, ROOT.parent.parent.parent]:
    if (p / 'mlcore').exists():
        ROOT = p
        break
ASSETS = ROOT / 'labs/lr-2/assets'
ASSETS.mkdir(parents=True, exist_ok=True)

## 1. Загрузка и анализ данных

In [ ]:
ds = load_lr2_dataset()
df = ds.features.copy()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Очистка: пробелы, конвертация в числа
exclude_cols = ['Ceramic Name', 'Part']
numeric_cols = [c for c in df.columns if c not in exclude_cols]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')

# Part — псевдо-метки для оценки (Body/Glaze)
y_true_labels, y_map = label_encode(df['Part'].values)
print(f'Part classes: {y_map}')
print(f'Missing values:\n{df[numeric_cols].isna().sum().loc[lambda s: s > 0]}')

# Убираем строки с пропусками и метаданные
df_clean = df[numeric_cols].dropna()
y_true_labels = y_true_labels[df[numeric_cols].notna().all(axis=1).values]

X_raw = df_clean.values.astype(np.float64)
X, mean, std = standardize(X_raw)
print(f'\nFinal X shape: {X.shape}')

## 2. Метрики кластеризации

- **Silhouette Score**: внутренняя метрика (не требует ground truth). Измеряет компактность и разделимость кластеров.
- **Adjusted Rand Index**: внешняя метрика. Сравниваем с `Part` (Body/Glaze) как псевдо-истиной.

In [ ]:
def silhouette_score(X: np.ndarray, labels: np.ndarray) -> float:
    """Mean silhouette coefficient."""
    n = len(X)
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2:
        return -1.0
    dists = np.linalg.norm(X[:, None] - X[None, :], axis=2)  # (n, n)
    scores = np.zeros(n)
    for i in range(n):
        same = labels == labels[i]
        same[i] = False
        if same.sum() == 0:
            scores[i] = 0.0
            continue
        a_i = dists[i, same].mean()
        b_i = np.inf
        for lbl in unique_labels:
            if lbl == labels[i]:
                continue
            other = labels == lbl
            if other.sum() > 0:
                b_i = min(b_i, dists[i, other].mean())
        scores[i] = (b_i - a_i) / max(a_i, b_i) if max(a_i, b_i) > 0 else 0.0
    return float(scores.mean())


def adjusted_rand_index(labels_true: np.ndarray, labels_pred: np.ndarray) -> float:
    """ARI from contingency table."""
    from itertools import product as iprod
    classes = np.unique(labels_true)
    clusters = np.unique(labels_pred)
    contingency = np.zeros((len(classes), len(clusters)), dtype=np.int64)
    cls_map = {c: i for i, c in enumerate(classes)}
    clu_map = {c: i for i, c in enumerate(clusters)}
    for t, p in zip(labels_true, labels_pred):
        contingency[cls_map[t], clu_map[p]] += 1
    a = contingency.sum(axis=1)
    b = contingency.sum(axis=0)
    n = len(labels_true)
    comb2 = lambda x: x * (x - 1) / 2
    index = sum(comb2(contingency[i, j]) for i, j in iprod(range(len(classes)), range(len(clusters))))
    expected = sum(comb2(ai) for ai in a) * sum(comb2(bj) for bj in b) / comb2(n) if comb2(n) > 0 else 0
    max_index = 0.5 * (sum(comb2(ai) for ai in a) + sum(comb2(bj) for bj in b))
    denom = max_index - expected
    return float((index - expected) / denom) if denom != 0 else 0.0

print('Metrics defined.')

## 3. K-means++

In [ ]:
def kmeans_pp(X: np.ndarray, k: int, max_iter: int = 300, tol: float = 1e-4,
              random_state: int = 42) -> tuple[np.ndarray, np.ndarray, int]:
    """K-means++ clustering."""
    rng = np.random.default_rng(random_state)
    n, d = X.shape

    # Init: D^2-weighted
    centroids = [X[rng.integers(n)]]
    for _ in range(k - 1):
        dists = np.min([np.sum((X - c) ** 2, axis=1) for c in centroids], axis=0)
        probs = dists / dists.sum()
        centroids.append(X[rng.choice(n, p=probs)])
    centroids = np.array(centroids)

    # Iterate
    for it in range(max_iter):
        dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        labels = dists.argmin(axis=1)
        new_centroids = np.array([X[labels == j].mean(axis=0) if (labels == j).sum() > 0
                                  else centroids[j] for j in range(k)])
        if np.max(np.linalg.norm(new_centroids - centroids, axis=1)) < tol:
            return labels, new_centroids, it + 1
        centroids = new_centroids

    return labels, centroids, max_iter

print('K-means++ defined.')

In [ ]:
# Подбор k: elbow + silhouette
ks = range(2, 9)
silhouettes = []
inertias = []

for k in ks:
    labels, centroids, iters = kmeans_pp(X, k)
    inertia = sum(np.sum((X[labels == j] - centroids[j]) ** 2) for j in range(k))
    inertias.append(inertia)
    sil = silhouette_score(X, labels)
    silhouettes.append(sil)
    print(f'k={k}: silhouette={sil:.3f}, inertia={inertia:.1f}, iters={iters}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(ks), inertias, 'o-')
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow')
ax2.plot(list(ks), silhouettes, 'o-')
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette'); ax2.set_title('Silhouette')
fig.tight_layout()
fig.savefig(ASSETS / 'elbow_silhouette.png', dpi=150)
plt.close(fig)
print('Saved elbow_silhouette.png')

In [ ]:
best_k = list(ks)[np.argmax(silhouettes)]
print(f'Best k by silhouette: {best_k}')

with timer('K-means++'):
    labels_km, centroids_km, iters_km = kmeans_pp(X, best_k)

sil_km = silhouette_score(X, labels_km)
ari_km = adjusted_rand_index(y_true_labels, labels_km)
print(f'K-means++ (k={best_k}): Silhouette={sil_km:.3f}, ARI={ari_km:.3f}')

## 4. DBSCAN

In [ ]:
def dbscan(X: np.ndarray, eps: float, min_samples: int) -> np.ndarray:
    """DBSCAN clustering. Returns labels (-1 = noise)."""
    n = len(X)
    dist_matrix = np.linalg.norm(X[:, None] - X[None, :], axis=2)
    labels = -np.ones(n, dtype=int)
    visited = np.zeros(n, dtype=bool)
    cluster_id = 0

    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        neighbors = np.where(dist_matrix[i] <= eps)[0]
        if len(neighbors) < min_samples:
            continue
        labels[i] = cluster_id
        seed = list(set(neighbors) - {i})
        while seed:
            j = seed.pop(0)
            if not visited[j]:
                visited[j] = True
                j_neighbors = np.where(dist_matrix[j] <= eps)[0]
                if len(j_neighbors) >= min_samples:
                    seed.extend([x for x in j_neighbors if not visited[x]])
            if labels[j] == -1:
                labels[j] = cluster_id
        cluster_id += 1

    return labels

print('DBSCAN defined.')

In [ ]:
# Grid search eps & min_samples
best_sil_db, best_params = -1, {}
for eps in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]:
    for ms in [3, 5, 7, 10]:
        labels_db = dbscan(X, eps, ms)
        n_clusters = len(set(labels_db) - {-1})
        n_noise = (labels_db == -1).sum()
        if n_clusters < 2:
            continue
        non_noise = labels_db != -1
        if non_noise.sum() < 5:
            continue
        sil = silhouette_score(X[non_noise], labels_db[non_noise])
        if sil > best_sil_db:
            best_sil_db = sil
            best_params = {'eps': eps, 'min_samples': ms, 'n_clusters': n_clusters, 'noise': n_noise}

print(f'Best DBSCAN: {best_params}, Silhouette={best_sil_db:.3f}')

with timer('DBSCAN'):
    labels_db_best = dbscan(X, best_params['eps'], best_params['min_samples'])

non_noise = labels_db_best != -1
ari_db = adjusted_rand_index(y_true_labels[non_noise], labels_db_best[non_noise])
print(f'DBSCAN ARI (non-noise): {ari_db:.3f}')

## 5. Визуализация (PCA -> 2D)

In [ ]:
# Manual PCA: 2 principal components
X_centered = X - X.mean(axis=0)
cov = X_centered.T @ X_centered / (len(X) - 1)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
top2 = eigenvectors[:, -2:][:, ::-1]
X_2d = X_centered @ top2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# K-means
for lbl in np.unique(labels_km):
    mask = labels_km == lbl
    ax1.scatter(X_2d[mask, 0], X_2d[mask, 1], s=20, alpha=0.7, label=f'Cluster {lbl}')
ax1.set_title(f'K-means++ (k={best_k})')
ax1.legend()

# DBSCAN
for lbl in sorted(set(labels_db_best)):
    mask = labels_db_best == lbl
    name = f'Cluster {lbl}' if lbl >= 0 else 'Noise'
    ax2.scatter(X_2d[mask, 0], X_2d[mask, 1], s=20, alpha=0.7, label=name,
               marker='x' if lbl == -1 else 'o')
ax2.set_title(f'DBSCAN (eps={best_params["eps"]}, min_samples={best_params["min_samples"]})')
ax2.legend()

fig.tight_layout()
fig.savefig(ASSETS / 'clusters_comparison.png', dpi=150)
plt.close(fig)
print('Saved clusters_comparison.png')

## 6. Сравнение

In [ ]:
comparison = pd.DataFrame([
    {'Алгоритм': 'K-means++', 'Кластеров': best_k, 'Silhouette': round(sil_km, 3), 'ARI': round(ari_km, 3)},
    {'Алгоритм': 'DBSCAN', 'Кластеров': best_params['n_clusters'], 'Silhouette': round(best_sil_db, 3), 'ARI': round(ari_db, 3)},
])
comparison

## 7. Выводы

1. K-means++ и DBSCAN реализованы с нуля на numpy.
2. Silhouette Score использован как основная метрика (не требует ground truth).
3. ARI по `Part` (Body/Glaze) позволяет оценить соответствие кластеров типам керамики.
4. Стандартизация критически важна из-за разных масштабов признаков (weight-% vs ppm).